In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path.cwd().resolve()
DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"
OUT = ROOT / "output"
OUT.mkdir(exist_ok=True)

fund = pd.read_csv(DATA_RAW / "fundamentals_358_raw.csv")
ret = pd.read_csv(DATA_PROCESSED / "return_panel_500.csv", parse_dates=["date"])
mom = pd.read_csv(DATA_PROCESSED / "momentum_panel_500.csv", parse_dates=["date"])
core = pd.read_csv(DATA_RAW / "universe_358.csv")

fund["report_date"] = pd.to_datetime(fund["report_date"])
fund = fund.sort_values(["ticker", "report_date"]).reset_index(drop=True)
ret["date"] = pd.to_datetime(ret["date"])
mom["date"] = pd.to_datetime(mom["date"])

core_tickers = core["ticker"].astype(str).tolist()
ret_core = ret[ret["ticker"].isin(core_tickers)].copy()
mom_core = mom[mom["ticker"].isin(core_tickers)].copy()

q = fund.copy()
q["book_to_sales"] = q["book_value"] / q["revenue"].replace(0, np.nan)
q["gross_profit_margin"] = (q["revenue"] - q["cogs"]) / q["revenue"].replace(0, np.nan)
q["net_margin"] = q["net_income"] / q["revenue"].replace(0, np.nan)
q["roe"] = q["net_income"] / q["book_value"].replace(0, np.nan)
q["assets_to_equity"] = q["total_assets"] / q["book_value"].replace(0, np.nan)
q = q.replace([np.inf, -np.inf], np.nan)

def zscore(s):
    sd = s.std(ddof=0)
    return (s - s.mean()) / sd if pd.notna(sd) and sd != 0 else np.nan

for c in ["book_to_sales", "gross_profit_margin", "net_margin", "roe", "assets_to_equity"]:
    q[c + "_z"] = q.groupby("report_date")[c].transform(zscore)

quality = q.groupby(["ticker", "report_date"], as_index=False).agg(
    book_to_sales=("book_to_sales_z", "mean"),
    gross_profit_margin=("gross_profit_margin_z", "mean"),
    net_margin=("net_margin_z", "mean"),
    roe=("roe_z", "mean"),
    assets_to_equity=("assets_to_equity_z", "mean")
)
quality["quality_score"] = quality[["gross_profit_margin", "net_margin", "roe", "assets_to_equity", "book_to_sales"]].mean(axis=1)

value = q.groupby(["ticker", "report_date"], as_index=False).agg(
    book_to_sales=("book_to_sales", "mean"),
    gross_profit_margin=("gross_profit_margin", "mean"),
    net_margin=("net_margin", "mean"),
    roe=("roe", "mean"),
    assets_to_equity=("assets_to_equity", "mean")
)
value["value_score"] = value.groupby("report_date")["book_to_sales"].transform(zscore)

fund["eps"] = fund["net_income"] / fund["shares_diluted"].replace(0, np.nan)
fund["eps_lag4"] = fund.groupby("ticker")["eps"].shift(4)
fund["sue"] = fund["eps"] - fund["eps_lag4"]
fund["sue_z"] = fund.groupby("report_date")["sue"].transform(zscore)
sue = fund[["ticker", "report_date", "sue", "sue_z"]].copy()

mom_core["momentum_12m_z"] = mom_core.groupby("date")["momentum_12m"].transform(zscore)

quality.to_csv(OUT / "factor_A_quality_358.csv", index=False)
value.to_csv(OUT / "factor_A_value_358.csv", index=False)
sue.to_csv(OUT / "factor_A_sue_358.csv", index=False)
mom_core.to_csv(OUT / "factor_A_momentum_358.csv", index=False)

coverage = pd.DataFrame({
    "model": ["A_core_358"],
    "n_tickers": [len(core_tickers)],
    "n_fund_rows": [len(fund)],
    "n_return_rows": [len(ret_core)],
    "n_momentum_rows": [len(mom_core)]
})
coverage.to_csv(OUT / "factor_A_coverage.csv", index=False)

print(coverage)
print(quality.head())
print(value.head())
print(sue.head())

        model  n_tickers  n_fund_rows  n_return_rows  n_momentum_rows
0  A_core_358        327         6146          48396            42875
  ticker report_date  book_to_sales  gross_profit_margin  net_margin  \
0      A  2020-07-31       0.679565            -0.383650    0.526567   
1      A  2020-10-31       0.512353            -0.330742    0.630155   
2      A  2021-01-31       0.329747            -0.376921   -0.025240   
3      A  2021-04-30       0.385610            -0.311588    0.259595   
4      A  2021-07-31       0.341430            -0.303175    0.149443   

        roe  assets_to_equity  quality_score  
0  0.137604          0.017508       0.195519  
1 -0.251313         -0.330030       0.046085  
2 -0.262200         -0.392551      -0.145433  
3 -0.235517         -0.265777      -0.033535  
4  0.137622          0.120456       0.089155  
  ticker report_date  book_to_sales  gross_profit_margin  net_margin  \
0      A  2020-07-31       3.950040             1.469469    0.157811   
1